[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crunchdao/numinous/blob/master/challenge/numinous/examples/quickstart.ipynb)

![Banner](https://raw.githubusercontent.com/crunchdao/quickstarters/refs/heads/master/competitions/numinous/assets/banner.webp)

# Numinous — Binary Event Forecasting

Predict the probability that real-world events resolve **"Yes"**.

Events are sourced from prediction markets and cover politics, crypto, sports, weather, and more.

## Scoring

Predictions are evaluated with the **Brier score**:

$$\text{Brier} = (p - o)^2$$

where $p$ is your predicted probability and $o \in \{0, 1\}$ is the actual outcome. **Lower is better.**

| Score | Meaning |
|-------|---------|
| 0.00  | Perfect |
| 0.25  | Uninformative (always 0.5) |
| 1.00  | Worst possible |

Predictions are clipped to $[0.01, 0.99]$. Missing predictions are scored as 0.25.

## Setup

In [48]:
#%pip install crunch-numinous
%pip install /Users/aboutrig/Documents/Pro/CrunchDao/Devs/crunch-numinous

Processing /Users/aboutrig/Documents/Pro/CrunchDao/Devs/crunch-numinous
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for crunch-numinous: filename=crunch_numinous-0.3.0-py3-none-any.whl size=489564 sha256=171b2cf403e32df7230644220fdf95acfffb0d90fe8859949751b657cb6ec169
  Stored in directory: /Users/aboutrig/Library/Caches/pip/wheels/ce/f6/92/ea8925abaa6c3981c5e5e9f31c9fda2995921c9b69aff76d58
Successfully built crunch-numinous
  Attempting uninstall: crunch-numinous
    Found existing installation: crunch-numinous 0.3.0
    Uninstalling crunch-numinous-0.3.0:
      Successfully uninstalled crunch-numinous-0.3.0

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Gateway

Your notebook has **no direct internet access** in production. All external calls (LLMs, search, OSINT...) go through the **gateway**.

- **In production**: `SANDBOX_PROXY_URL` is set automatically — API costs are covered by CrunchDAO
- **Locally**: you run the gateway yourself with your own API keys — most providers offer a free tier

See the [API Reference](https://github.com/crunchdao/numinous/blob/main/numinous/gateway/API_REFERENCE.md) for all available endpoints.

In [49]:
# --- LOCAL TESTING ONLY — remove this cell before submitting ---
# Option 1: set your API keys here
import os
#os.environ["OPENAI_API_KEY"] = ""           # https://platform.openai.com/api-keys
# os.environ["OPENROUTER_API_KEY"] = "sk-or-..."  # https://openrouter.ai/settings/keys
# os.environ["CHUTES_API_KEY"] = "..."            # https://chutes.ai/app

# Option 2: put them in a persistent env file (one KEY=VALUE per line)
# You can create it manually or run: crunch-numinous gateway configure
from pathlib import Path
print(f"Env file location: {Path.home() / '.crunch-numinous-gateway.env'}")

Env file location: /Users/aboutrig/.crunch-numinous-gateway.env


In [50]:
#!crunch-numinous gateway restart
# To see HTTP requests and responses in the logs, use:
!crunch-numinous gateway restart --debug


Stopping gateway (PID: 96962)...
Gateway stopped


╭─────────────────────────────╮
│                             │
│  Starting Numinous Gateway  │
│                             │
╰─────────────────────────────╯

API keys set: OPENAI_API_KEY
Missing API keys: CHUTES_API_KEY, DESEARCH_API_KEY, PERPLEXITY_API_KEY, 
VERICORE_API_KEY, OPENROUTER_API_KEY
You can set them via environment variables or run: crunch-numinous gateway 
configure

  Starting gateway... ✓

╭───────────────── Gateway Running ──────────────────╮
│ Gateway started!                                   │
│                                                    │
│ PID: 97005                                         │
│ URL: http://localhost:8090                         │
│ Logs: /Users/aboutrig/.crunch-numinous-gateway.log │
│                                                    │
│ View logs: crunch-numinous gateway logs            │
│ Stop: crunch-numinous gateway stop                 │
╰────────────────────────────────────────

## Your Model

Subclass `TrackerBase` and implement `_predict()`.

```
Input:  {"event_id": "...", "title": "Will X happen?", "yes_price": 0.65, ...}
Output: {"event_id": "...", "prediction": 0.72}
```

This example asks an LLM to estimate P(Yes) given the event details and current market price. It calls the gateway's OpenAI endpoint — swap the model or provider to experiment.

In [51]:
import os
import re

import httpx
from numinous.tracker import TrackerBase

GATEWAY_URL = os.environ.get("SANDBOX_PROXY_URL", "http://localhost:8090")


class LLMTracker(TrackerBase):
    """Asks an LLM to estimate P(Yes) for each event."""

    def _predict(self, subject, resolve_horizon_seconds, step_seconds):
        data = self._get_data(subject)
        if not isinstance(data, dict):
            return {"event_id": subject, "prediction": 0.5}

        event_id = data.get("event_id", subject)
        title = data.get("title", "")
        description = data.get("description", "")
        yes_price = float(data.get("yes_price", 0.5))

        prompt = (
            "You are a forecasting expert. Estimate the probability that "
            "this event resolves Yes.\n\n"
            f"Event: {title}\n"
            f"Description: {description}\n"
            f"Current market price: {yes_price}\n\n"
            "Reply with ONLY a decimal number between 0 and 1."
        )

        try:
            resp = httpx.post(
                f"{GATEWAY_URL}/api/gateway/openai/responses",
                json={
                    "model": "gpt-4.1-nano",
                    "input": [{"role": "user", "content": prompt}],
                    "temperature": 0.2,
                    "max_output_tokens": 16,
                },
                timeout=180.0,
            )
            resp.raise_for_status()
            result = resp.json()

            # Extract text from OpenAI response
            text = ""
            for item in result.get("output", []):
                for content in (item.get("content") or []):
                    if content.get("text"):
                        text = content["text"]

            match = re.search(r"(0\.\d+|1\.0|0|1)", text)
            prediction = float(match.group(1)) if match else yes_price
        except Exception as e:
            raise
            #print(f"  [{event_id}] LLM call failed: {e}, falling back to market price")
            #prediction = yes_price

        return {"event_id": event_id, "prediction": max(0.0, min(1.0, prediction))}

## Test Locally

Feed sample events and check predictions.

In [52]:
SAMPLE_EVENTS = [
    {"event_id": "evt-btc-100k", "title": "Will BTC exceed $100k by end of March?",
     "yes_price": 0.65, "cutoff": "2026-04-01T00:00:00Z", "source": "polymarket",
     "description": "", "volume_24h": 150000.0, "metadata": {}},
    {"event_id": "evt-rain-nyc", "title": "Will it rain in NYC tomorrow?",
     "yes_price": 0.80, "cutoff": "2026-03-25T00:00:00Z", "source": "polymarket",
     "description": "", "volume_24h": 5000.0, "metadata": {}},
    {"event_id": "evt-fed-rate", "title": "Will the Fed cut rates in April?",
     "yes_price": 0.35, "cutoff": "2026-04-30T00:00:00Z", "source": "polymarket",
     "description": "", "volume_24h": 500000.0, "metadata": {}},
]

OUTCOMES = {"evt-btc-100k": 1, "evt-rain-nyc": 1, "evt-fed-rate": 0}

In [53]:
model = LLMTracker()
total_brier = 0.0

print(f"{'Event':50s} {'Market':>7s} {'LLM':>6s} {'Out':>5s} {'Brier':>8s}")
print("-" * 80)

for event in SAMPLE_EVENTS[0:1]:
    model.feed_update(event)
    result = model.predict(event["event_id"], resolve_horizon_seconds=3600, step_seconds=1)

    p = max(0.01, min(0.99, result["prediction"]))
    o = OUTCOMES[event["event_id"]]
    brier = (p - o) ** 2
    total_brier += brier

    print(f"{event['title'][:50]:50s} {event['yes_price']:7.2f} {p:6.3f} {'Yes' if o else 'No':>5s} {brier:8.4f}")

print("-" * 80)
print(f"{'Average Brier':50s} {'':>7s} {'':>6s} {'':>5s} {total_brier / len(SAMPLE_EVENTS):8.4f}")
print(f"{'Baseline (always 0.5)':50s} {'':>7s} {'':>6s} {'':>5s} {'0.2500':>8s}")

Event                                               Market    LLM   Out    Brier
--------------------------------------------------------------------------------


HTTPStatusError: Client error '401 Unauthorized' for url 'http://localhost:8090/api/gateway/openai/responses'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

In [54]:
!crunch-numinous gateway logs --no-follow

INFO:     127.0.0.1:51296 - "POST /api/gateway/openai/responses HTTP/1.1" 422 Unprocessable Content
INFO:     127.0.0.1:52001 - "GET /api/health HTTP/1.1" 200 OK
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [95493]
No .env file found. Make sure to create a .env file in the gateway/ directory and add your API keys for Chutes and Desearch.
INFO:     Started server process [96571]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8090 (Press CTRL+C to quit)
INFO:     127.0.0.1:52027 - "GET /api/health HTTP/1.1" 200 OK
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [96571]
No .env file found. Make sure to create a .env file in the gateway/ directory and add your API keys for Chutes and Desearch.
INFO:     Started server pro

# Submit your Notebook

To submit your work, you must:
1. Download your Notebook from Colab
2. Upload it to the platform
3. Deploy it in real condition

### >> https://hub.crunchdao.com/competitions/numinous/submit/notebook

The platform will find your `TrackerBase` subclass automatically.

![Download and Submit Notebook](https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/documentation/animations/download-and-submit-notebook-deployment.gif)